In [2]:
import pandas as pd
import numpy as np


In [7]:
# Load the movies.csv dataset
df = pd.read_csv('movies.csv')

# Display the first 5 rows of the DataFrame
print("First 5 rows of the dataset:")
display(df.head(10))

First 5 rows of the dataset:


,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross
0,Blood Red Sky,(2021),"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN
1,Masters of the Universe: Revelation,(2021– ),"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN
2,The Walking Dead,(2010–2022),"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN
3,Rick and Morty,(2013– ),"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN
4,Army of Thieves,(2021),"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN
5,Outer Banks,(2020– ),"\nAction, Crime, Drama",7.6,\nA group of teenagers from the wrong side of ...,"\n \n Stars:\nChase Stokes, \nMa...","25,858",50.0,NaN
6,The Last Letter from Your Lover,(2021),"\nDrama, Romance",6.8,\nA pair of interwoven stories set in the past...,\n Director:\nAugustine Frizzell\n| \n S...,"5,283",110.0,NaN
7,Dexter,(2006–2013),"\nCrime, Drama, Mystery",8.6,"\nBy day, mild-mannered Dexter is a blood-spat...","\n \n Stars:\nMichael C. Hall, \...","665,387",53.0,NaN
8,Never Have I Ever,(2020– ),\nComedy,7.9,\nThe complicated life of a modern-day first g...,\n \n Stars:\nMaitreyi Ramakrish...,"34,530",30.0,NaN
9,Virgin River,(2019– ),"\nDrama, Romance",7.4,"\nSeeking a fresh start, nurse practitioner Me...",\n \n Stars:\nAlexandra Breckenr...,"27,279",44.0,NaN


### Dataset Information
Let's check the number of rows and columns, data types, and non-null values.

In [4]:
# Display basic information about the dataset
print("Dataset Info:")
df.info()

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MOVIES    9999 non-null   object 
 1   YEAR      9355 non-null   object 
 2   GENRE     9919 non-null   object 
 3   RATING    8179 non-null   float64
 4   ONE-LINE  9999 non-null   object 
 5   STARS     9999 non-null   object 
 6   VOTES     8179 non-null   object 
 7   RunTime   7041 non-null   float64
 8   Gross     460 non-null    object 
dtypes: float64(2), object(7)
memory usage: 703.2+ KB


### Descriptive Statistics
Now, let's look at the descriptive statistics for numerical columns.

In [5]:
# Display descriptive statistics
print("Descriptive Statistics:")
display(df.describe())

Descriptive Statistics:


,RATING,RunTime
count,8179.000000,7041.000000
mean,6.921176,68.688539
std,1.220232,47.258056
min,1.100000,1.000000
25%,6.200000,36.000000
50%,7.100000,60.000000
75%,7.800000,95.000000
max,9.900000,853.000000


### Data Type Conversion and Cleaning (YEAR, VOTES, Gross)
To effectively handle missing values, we first need to clean and convert 'YEAR', 'VOTES', and 'Gross' columns to numerical types. This involves extracting numerical data from their string representations and splitting 'YEAR' into start and end years.

In [16]:
import re

def parse_year_range(year_str):
    if pd.isna(year_str):
        return np.nan, np.nan

    year_str = str(year_str).strip()
    # Remove parentheses for easier parsing
    year_str = year_str.replace('(', '').replace(')', '')

    if '–' in year_str: # Handles ranges like '2010–2022' or '2021– '
        parts = year_str.split('–')
        start_year = int(parts[0].strip()) if parts[0].strip() else np.nan
        end_year = int(parts[1].strip()) if parts[1].strip() else np.nan
    else: # Handles single years like '2021'
        start_year = int(year_str.strip()) if year_str.strip() else np.nan
        end_year = start_year # If it's a single year, start and end are the same

    return start_year, end_year

# Apply the function to create new columns
df[['START_YEAR', 'END_YEAR']] = df['YEAR'].apply(lambda x: pd.Series(parse_year_range(x)))

# Convert new columns to integer type, allowing for NaNs
df['START_YEAR'] = df['START_YEAR'].astype('Int64')
df['END_YEAR'] = df['END_YEAR'].astype('Int64')

# Drop the original 'YEAR' column as it's been split
df = df.drop(columns=['YEAR'])

# Clean and convert 'VOTES' column
# Remove commas and convert to float
df['VOTES'] = df['VOTES'].str.replace(',', '', regex=False).astype(float)

# Clean and convert 'Gross' column
# Remove '$', 'M', and commas, then convert to float. Assume 'M' means millions.
def clean_gross(gross_str):
    if pd.isna(gross_str):
        return np.nan
    gross_str = str(gross_str).replace('$', '').replace(',', '')
    if 'M' in gross_str:
        try:
            return float(gross_str.replace('M', '')) * 1_000_000
        except ValueError:
            return np.nan
    else:
        try:
            return float(gross_str)
        except ValueError:
            return np.nan

df['Gross'] = df['Gross'].apply(clean_gross)

print("Data types after cleaning 'START_YEAR', 'END_YEAR', 'VOTES', 'Gross':")
df.info()

print("\nFirst 5 rows after cleaning and conversion:")
display(df.head())

ValueError: invalid literal for int() with base 10: 'I 2018'

### Handling Missing Values: Dropping Rows with NaNs
Now that the relevant columns have been converted to appropriate data types, we can proceed with dropping rows that contain any NaN values. We will first check the number of missing values and the DataFrame shape before and after the operation.

In [ ]:
# Display the number of missing values before dropping
print("Missing values before dropping NaNs:")
display(df.isnull().sum()[df.isnull().sum() > 0])

# Store the original shape of the DataFrame
original_shape = df.shape
print(f"\nOriginal DataFrame shape: {original_shape}")

# Drop rows with any NaN values
df_cleaned = df.dropna().copy()

# Display the new shape of the DataFrame
cleaned_shape = df_cleaned.shape
print(f"Cleaned DataFrame shape after dropping NaNs: {cleaned_shape}")
print(f"Number of rows dropped: {original_shape[0] - cleaned_shape[0]}")

# Display the info of the cleaned DataFrame to confirm no more NaNs
print("\nInfo of cleaned DataFrame:")
df_cleaned.info()

# Display the first few rows of the cleaned DataFrame
print("\nFirst 5 rows of the cleaned DataFrame:")
display(df_cleaned.head())

In [19]:
df[df['YEAR']== '2018']

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross


In [15]:
for x in df['YEAR']:
  print(str(x))

(2021)
(2021– )
(2010–2022)
(2013– )
(2021)
(2020– )
(2021)
(2006–2013)
(2020– )
(2019– )
(2021)
(2016–2021)
(2021)
(2021– )
(2011– )
(2005– )
(2008–2013)
(2017– )
(2017–2021)
(2016– )
(2021)
(1994–2004)
(2014– )
(2013– )
(2021– )
(2021)
(2013–2020)
(2015– )
(2005–2020)
(2013– )
(2021)
(2016– )
(2013–2022)
(2003– )
(2019– )
(2009–2020)
(I) (2018– )
(2019– )
(2010–2015)
(2013–2022)
(2011–2019)
(2017–2021)
(2015–2020)
(2014– )
(2015– )
(2011– )
(2017– )
(2016– )
(2021– )
(2021)
(2014– )
(2005–2014)
(2009–2015)
(2008–2014)
(2016–2018)
(2009–2017)
(2020)
(2019– )
(2018–2021)
(2017–2020)
(2020)
(2016– )
(1987–1994)
(2021)
(2021)
(2019– )
(2015– )
(2018– )
(2012– )
(2014–2020)
(2011–2018)
(2005–2017)
(2019– )
(2021)
(2021– )
(2021– )
(2021– )
(2017)
(2007–2015)
(2018– )
(2000–2007)
(2021– )
(2021– )
(2021)
(II) (2007– )
(1993)
(1999–2022)
(2015–2018)
(2017–2020)
(2016– )
(2018– )
(2019– )
(2014–2019)
(2016– )
(2013–2020)
(2016)
(2021)
(2012–2020)
(2013–2019)
(2007–2012)
(2020– )
(2018–2021)


AttributeError: 'DataFrame' object has no attribute 'isNaN'